# Check 3: Metrics with known answers

Small artificial states have answers that can be calculated by hand, so they can reveal a wrong formula even when a simulation looks plausible.

For unit headings $d_i$, polarization is $P = \|N^{-1}\sum_i d_i\|$. In two dimensions, rotation is $R = |N^{-1}\sum_i (\hat r_i \times d_i)|$, where $r_i$ points from the group centroid to fish $i$. These order parameters follow [Tunstrøm et al. (2013)](https://doi.org/10.1371/journal.pcbi.1002915).

All configurations below are fixed snapshots. They test measurement, not whether the social dynamics spontaneously generate these arrangements.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shiner_model.py").is_file()
)
MODEL_DIR = PROJECT_ROOT
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))

import numpy as np
import pandas as pd
from shiner_model import GoldenShinersModel, compute_group_metrics

TOLERANCE = 1e-12

from types import SimpleNamespace


def metric_fixture(positions, directions, speeds=None):
    """Construct a fixed fish state without running social dynamics."""
    positions = np.asarray(positions, dtype=float)
    directions = np.asarray(directions, dtype=float)
    assert positions.shape == directions.shape
    assert positions.ndim == 2 and positions.shape[1] == 2
    np.testing.assert_allclose(np.linalg.norm(directions, axis=1), 1.0, atol=TOLERANCE)
    if speeds is None:
        speeds = np.ones(len(positions))
    assert len(speeds) == len(positions)
    return SimpleNamespace(agents=[
        SimpleNamespace(position=position, direction=direction, speed=float(speed))
        for position, direction, speed in zip(positions, directions, speeds)
    ])


metric_checks = []

def check_metric(case, metrics, name, expected):
    actual = metrics[name]
    np.testing.assert_allclose(actual, expected, atol=TOLERANCE, rtol=TOLERANCE, equal_nan=True)
    metric_checks.append({
        "Case": case, "Metric": name,
        "Expected": expected, "Observed": actual, "Result": "passed",
    })


## Aligned and cancelling headings

Four fish at the corners of a square all point east. Their mean heading is `(1, 0)`, so polarization is 1. Their signed contributions to rotation cancel, so rotation is 0. The nearest neighbour is 2 distance units away, and each fish is $\sqrt{2}$ units from the centroid.

A second configuration uses balanced north, south, east and west headings. These headings cancel exactly, giving polarization 0. This avoids assuming that an arbitrary finite random sample must have exactly zero polarization.


In [2]:
square_positions = [(-1, -1), (-1, 1), (1, -1), (1, 1)]
aligned = compute_group_metrics(metric_fixture(square_positions, [(1, 0)] * 4))
for name, expected in {
    "Polarization": 1.0, "Rotation": 0.0, "MeanSpeed": 1.0,
    "MeanNND": 2.0, "MeanDistanceToCentroid": np.sqrt(2),
    "CentroidX": 0.0, "CentroidY": 0.0,
}.items():
    check_metric("Aligned square", aligned, name, expected)

balanced = compute_group_metrics(metric_fixture(
    square_positions, [(1, 0), (-1, 0), (0, 1), (0, -1)]
))
check_metric("Balanced headings", balanced, "Polarization", 0.0)

pd.DataFrame(metric_checks)


,Case,Metric,Expected,Observed,Result
0,Aligned square,Polarization,1.000000,1.000000,passed
1,Aligned square,Rotation,0.000000,0.000000,passed
2,Aligned square,MeanSpeed,1.000000,1.000000,passed
3,Aligned square,MeanNND,2.000000,2.000000,passed
4,Aligned square,MeanDistanceToCentroid,1.414214,1.414214,passed
5,Aligned square,CentroidX,0.000000,0.000000,passed
6,Aligned square,CentroidY,0.000000,0.000000,passed
7,Balanced headings,Polarization,0.000000,0.000000,passed


**Interpretation.** Perfect alignment gives polarization 1, and balanced opposing headings give polarization 0. The distance checks also agree with the known square geometry. Low polarization alone describes cancellation of headings; it does not establish that a group is compact or biologically realistic.


## Circular motion in both directions

Place four fish on a unit-radius circle at the cardinal points, with tangent headings. For counterclockwise motion each signed cross product is +1; for clockwise motion it is −1. Taking the absolute mean gives rotation 1 in both cases. The headings cancel, so polarization is 0.

The metric intentionally measures rotational order without distinguishing clockwise from counterclockwise circulation.


In [3]:
circle_positions = [(1, 0), (0, 1), (-1, 0), (0, -1)]
ccw_headings = [(0, 1), (-1, 0), (0, -1), (1, 0)]
cw_headings = [(-dx, -dy) for dx, dy in ccw_headings]

circulation_results = []
for case, headings, expected_signed in [
    ("Counterclockwise circle", ccw_headings, 1.0),
    ("Clockwise circle", cw_headings, -1.0),
]:
    measured = compute_group_metrics(metric_fixture(circle_positions, headings))
    for name, expected in {
        "Polarization": 0.0, "Rotation": 1.0,
        "MeanNND": np.sqrt(2), "MeanDistanceToCentroid": 1.0,
        "CentroidX": 0.0, "CentroidY": 0.0,
    }.items():
        check_metric(case, measured, name, expected)
    signed_mean = np.mean([
        x * dy - y * dx
        for (x, y), (dx, dy) in zip(circle_positions, headings)
    ])
    np.testing.assert_allclose(signed_mean, expected_signed)
    circulation_results.append({
        "Case": case, "Signed circulation": signed_mean,
        "Polarization": measured["Polarization"], "Rotation": measured["Rotation"],
    })

# Equal clockwise/counterclockwise contributions can cancel in the group mean.
mixed_headings = [(0, 1), (1, 0), (0, -1), (-1, 0)]
mixed = compute_group_metrics(metric_fixture(circle_positions, mixed_headings))
check_metric("Opposing circular contributions", mixed, "Polarization", 0.0)
check_metric("Opposing circular contributions", mixed, "Rotation", 0.0)
circulation_results.append({
    "Case": "Opposing circular contributions", "Signed circulation": 0.0,
    "Polarization": mixed["Polarization"], "Rotation": mixed["Rotation"],
})
pd.DataFrame(circulation_results)


,Case,Signed circulation,Polarization,Rotation
0,Counterclockwise circle,1.0,0.0,1.0
1,Clockwise circle,-1.0,0.0,1.0
2,Opposing circular contributions,0.0,0.0,0.0


**Interpretation.** Both coherent circles return rotation 1 and polarization 0, as expected. Equal contributions in opposite rotational directions cancel to rotation 0. A low global rotation value therefore does not imply that every individual has zero tangential motion.


## Compactness, spacing, translation and rotation

Scale the same circle from radius 0.25 to radius 4. The mean distance to the centroid and mean nearest-neighbour distance must both increase by a factor of 16. Polarization and rotation should remain unchanged because they use unit directions rather than group size.

Translating the group changes only the centroid coordinates. Rotating both positions and headings together should preserve all scalar metrics. These checks catch accidental dependence on the arena origin, group size, or the chosen coordinate axes.


In [4]:
compactness_results = []
for radius in (0.25, 4.0):
    positions = np.asarray(circle_positions) * radius + np.array([10.0, 10.0])
    measured = compute_group_metrics(metric_fixture(positions, ccw_headings))
    case = f"Circle radius={radius}"
    for name, expected in {
        "Polarization": 0.0, "Rotation": 1.0,
        "MeanDistanceToCentroid": radius, "MeanNND": radius * np.sqrt(2),
        "CentroidX": 10.0, "CentroidY": 10.0,
    }.items():
        check_metric(case, measured, name, expected)
    compactness_results.append({"Radius": radius, **measured})
compactness_results = pd.DataFrame(compactness_results)
np.testing.assert_allclose(
    compactness_results.loc[1, ["MeanNND", "MeanDistanceToCentroid"]].to_numpy(dtype=float)
    / compactness_results.loc[0, ["MeanNND", "MeanDistanceToCentroid"]].to_numpy(dtype=float),
    16.0,
)

base = compute_group_metrics(metric_fixture(circle_positions, ccw_headings))
translated = compute_group_metrics(metric_fixture(
    np.asarray(circle_positions) + (8.0, 9.0), ccw_headings
))
angle = np.pi / 7
rotation_matrix = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
rotated = compute_group_metrics(metric_fixture(
    np.asarray(circle_positions) @ rotation_matrix.T,
    np.asarray(ccw_headings) @ rotation_matrix.T,
))
for name in ("Polarization", "Rotation", "MeanSpeed", "MeanNND", "MeanDistanceToCentroid"):
    check_metric("Translated circle", translated, name, base[name])
    check_metric("Rotated coordinates", rotated, name, base[name])
check_metric("Translated circle", translated, "CentroidX", 8.0)
check_metric("Translated circle", translated, "CentroidY", 9.0)

compactness_results[["Radius", "Polarization", "Rotation", "MeanNND", "MeanDistanceToCentroid"]]


,Radius,Polarization,Rotation,MeanNND,MeanDistanceToCentroid
0,0.25,0.0,1.0,0.353553,0.25
1,4.00,0.0,1.0,5.656854,4.00


**Interpretation.** Expanding the group changes both distance measures by the expected factor while leaving its directional order unchanged. Translation and rotation preserve the scalar metrics. This separates compactness from alignment and circulation, which is useful when a later light field changes where the fish accumulate.


## Nearest neighbours, variable speeds and degenerate geometry

A 3–4–5 triangle gives a nearest-neighbour distance of exactly 5 for two fish. Speeds 0.5 and 1.5 have mean 1. A single fish has no other fish to serve as its nearest neighbour, so `MeanNND = NaN` is intentional.

A fish exactly at the centroid has no defined radial unit vector. The implementation assigns it a zero rotation contribution, avoiding division by zero. Two distinct fish at the same position legitimately have nearest-neighbour distance 0; this is different from accidentally counting a fish as its own neighbour.


In [5]:
pair = compute_group_metrics(metric_fixture(
    [(0, 0), (3, 4)], [(1, 0), (1, 0)], speeds=[0.5, 1.5]
))
for name, expected in {
    "MeanNND": 5.0, "MeanSpeed": 1.0,
    "MeanDistanceToCentroid": 2.5, "CentroidX": 1.5, "CentroidY": 2.0,
}.items():
    check_metric("Two fish at distance 5", pair, name, expected)

single = compute_group_metrics(metric_fixture([(2, 3)], [(0, 1)]))
for name, expected in {
    "Polarization": 1.0, "Rotation": 0.0, "MeanNND": np.nan,
    "MeanDistanceToCentroid": 0.0, "CentroidX": 2.0, "CentroidY": 3.0,
}.items():
    check_metric("Single fish", single, name, expected)

centroid_fish = compute_group_metrics(metric_fixture(
    [(-1, 0), (0, 0), (1, 0)], [(0, 1)] * 3
))
for name, expected in {
    "Polarization": 1.0, "Rotation": 0.0,
    "MeanNND": 1.0, "MeanDistanceToCentroid": 2 / 3,
}.items():
    check_metric("Fish at centroid", centroid_fish, name, expected)
assert all(np.isfinite(centroid_fish[name]) for name in (
    "Polarization", "Rotation", "MeanNND", "MeanDistanceToCentroid"
))

overlapping = compute_group_metrics(metric_fixture(
    [(0, 0), (0, 0), (3, 4)], [(1, 0)] * 3
))
check_metric("Distinct fish at the same position", overlapping, "MeanNND", 5 / 3)

metric_check_table = pd.DataFrame(metric_checks)
print(f"{len(metric_check_table)} known-answer comparisons passed.")
metric_check_table


62 known-answer comparisons passed.


,Case,Metric,Expected,Observed,Result
0,Aligned square,Polarization,1.000000,1.000000,passed
1,Aligned square,Rotation,0.000000,0.000000,passed
2,Aligned square,MeanSpeed,1.000000,1.000000,passed
3,Aligned square,MeanNND,2.000000,2.000000,passed
4,Aligned square,MeanDistanceToCentroid,1.414214,1.414214,passed
...,...,...,...,...,...
57,Fish at centroid,Polarization,1.000000,1.000000,passed
58,Fish at centroid,Rotation,0.000000,0.000000,passed
59,Fish at centroid,MeanNND,1.000000,1.000000,passed
60,Fish at centroid,MeanDistanceToCentroid,0.666667,0.666667,passed


**Interpretation.** Nearest-neighbour distance excludes the fish itself, speed is averaged correctly, and centroid geometry does not produce numerical errors. A single-fish `NaN` has a clear mathematical meaning and should not be replaced by zero. Geometric overlap can be measured without a division error; whether the social rules resolve it is checked separately in `data_collector.ipynb`.

**Outcome of check 3.** All listed hand-calculable answers and geometric invariances pass. This supports the implementation of polarization, rotation, centroid, speed and spacing measures. It does not validate a particular parameter calibration or prove that the simulated fish reproduce an experiment.

**Reference:** [Tunstrøm et al. (2013), *Collective States, Multistability and Transitional Behavior in Schooling Fish*](https://doi.org/10.1371/journal.pcbi.1002915). The numerical edge-case conventions and test geometries are implementation choices.
